In [1]:
# This notebook demonstrates how to use ROSALIA to analyze stray light in a simple exposure. 
import rosalia as rs
# import os
# os.system("cp -v /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933_BAK/MAST/Roman/r0102001002001005001_0004_wfi*.asdf /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/" )

path = "/Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/r0102001002001005001_0004_wfi*_f158_cal.asdf"
custom_roman_exposure = rs.core.exposure(filename=path) 
# straylight_out = custom_roman_exposure.straylight(verbose=True)

100%|██████████| 18/18 [00:04<00:00,  3.76it/s]


In [12]:
self = custom_roman_exposure
# def make_mosaic(arrays, wcs):

def generate_mosaic(data, astropywcs, resolution=None):
    import astropy.units as u
    from reproject import reproject_interp
    from reproject.mosaicking import reproject_and_coadd, find_optimal_celestial_wcs

    input_data_for_reproject = list(zip(data, astropywcs))

    if resolution is not None:
        resolution = resolution*u.arcsec

    optimal_wcs = find_optimal_celestial_wcs(input_data=input_data_for_reproject, 
                                            resolution=resolution)

    output = reproject_and_coadd(input_data=input_data_for_reproject, 
                                output_projection=optimal_wcs[0], 
                                shape_out=optimal_wcs[1],
                                reproject_function=reproject_interp,
                                #progress_bar=True,
                                intermediate_memmap=True)
    output[0].data[np.isnan(output[0].data)] = np.nan
    return(output[0], optimal_wcs[0].to_header())

mos_data, mos_wcs = generate_mosaic(data=self.DATA, astropywcs=self.ASTROPYWCS, resolution=0.11)



NameError: name 'np' is not defined

In [ ]:
rs.utils.save_fits(mos_data, "test.fits", mos_wcs)

'test.fits'

In [ ]:
! ls

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14,14))
im = ax.imshow(output[0], origin="lower", vmin=0, vmax=5)
plt.colorbar(im, ax=ax)

In [ ]:
print(optimal_wcs[0].array_shape)

In [ ]:

import glob
import asdf
list_of_files = glob.glob(path)
print(list_of_files)
print(len(list_of_files))

input_asdf = asdf.open(list_of_files[0])
input_asdf["roman"]["meta"]['pointing']

In [ ]:
type(custom_roman_exposure.ASTROPYWCS[0])

In [ ]:
import numpy as np
cd = w.wcs.cd
pa_rad = np.arctan2(cd[1,0], cd[0,0])
pa_deg = np.degrees(pa_rad)
pa_deg

In [ ]:
import numpy as np
w = custom_roman_exposure.ASTROPYWCS[10]
# Retrieve the full coordinate transformation matrix
cd_matrix = w.pixel_scale_matrix  # Returns combined PC/CD matrix scaled by CDELT

# cd_matrix is [[cd1_1, cd1_2], 
#              [cd2_1, cd2_2]]
# For celestial coordinates: axis 0 is RA, axis 1 is Dec

# PA of pixel +Y axis (East of North):
pa_rad = np.arctan2(cd_matrix[0, 1], cd_matrix[1, 1])
pa_deg = np.degrees(pa_rad) % 360

print(f"Position Angle: {pa_deg:.2f} deg")

In [ ]:
self = custom_roman_exposure

In [ ]:
dir(self)

In [ ]:
# def make_mosaic():
# self.
# /Users/aborlaff/NASA/ROSALIA_DEPOT/ndi_lvl1/lvl1_SCA_1_SUB_X-2.56_Y-2.56.fits

In [ ]:
ndi_lvl1

In [ ]:
# Reconstruct the NDI map
def overlay_NDI_map(ra_point, dec_point, pa_point, SCA, ndi_lvl, xlabel="2.56", ylabel="2.56"):

    # Load the NDI map 
    ndi_name = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl" + str(ndi_lvl) + "/lvl" + str(ndi_lvl) + "_SCA_" + str(SCA) + ".pkl"
    with open(ndi_name, "rb") as f:
        ndi_dict = pickle.load(f)

    # Get the right superpixel
    label = "X" + xlabel + "_Y" + ylabel 
    i = ndi_dict["NDI_labels"].index(label)

    # Get the original array. 
    original_grid = ndi_lvl1["ndi_interpolator"][i].grid          # list of arrays for each dimension
    original_shape = tuple(len(p) for p in original_grid)
    reconstructed_array = np.flip(ndi_lvl1["ndi_interpolator"][i].values.reshape(original_shape).T, axis=1)
    # return(reconstructed_array)
    print(list(ndi_dict))
    w = ndi_dict["wcs"][i]
    w.wcs.crota = -pa_point,-pa_point
    w.wcs.crval = ra_point,dec_point

    header = w.to_header()
    print(header)

    rs.utils.save_fits(array=reconstructed_array, name="test_ndi.fits", header=header,  overwrite=True)

    return(reconstructed_array)
    # x_stars, y_stars = w.wcs_world2pix(ra_stars, dec_stars, 0)
    # return x_stars, y_stars


In [ ]:
ndi = overlay_NDI_map(ra_point=self.RA_TARG, dec_point=self.DEC_TARG, pa_point=self.PA, SCA=1, ndi_lvl=1, xlabel="2.56", ylabel="2.56")
plt.imshow(ndi)

In [ ]:
self.PA

In [ ]:
import numpy as np
from astropy.io import fits
# Load original 
# overlay_NDI_map(ndi_map, ra_point, dec_point, pa_point) 
SCA=6 
import pickle 
if True:

    ndi_name_1 = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl1/lvl1_SCA_" + str(SCA) + ".pkl"
    ndi_name_2 = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl2/lvl2_SCA_" + str(SCA) + ".pkl"
    with open(ndi_name_1, "rb") as f:
        ndi_lvl1 = pickle.load(f)
    with open(ndi_name_2, "rb") as f:
        ndi_lvl2 = pickle.load(f)

"""
        w = ndi_wcs
        w.wcs.crota = -pa_point,-pa_point
        w.wcs.crval = ra_point,dec_point
        x_stars, y_stars = w.wcs_world2pix(ra_stars, dec_stars, 0))
"""
ndi_lvl1



ndi_ori = fits.open("/Users/aborlaff/NASA/ROSALIA_DEPOT/ndi_lvl1/lvl1_SCA_6_SUB_X-17.92_Y-17.92_TAN.fits")[0].data

i = 0

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(reconstructed_array, origin='lower', cmap='viridis', vmin=0, vmax=0.5)

ax.imshow(reconstructed_array-ndi_ori, origin='lower', cmap='viridis')
